In [65]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"

orders_path = DATA_DIR / "orders.csv"
order_products_path = DATA_DIR / "order_products.csv"
products_path = DATA_DIR / "products.csv"
aisles_path = DATA_DIR / "aisles.csv"
departments_path = DATA_DIR / "departments.csv"

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)

Base directory: c:\Users\Admin\Storage\Akademik\Dersler\Erasmus Classes\Data Mining\Assignment1
Data directory: c:\Users\Admin\Storage\Akademik\Dersler\Erasmus Classes\Data Mining\Assignment1\data


In [66]:
orders = pd.read_csv(orders_path)
products = pd.read_csv(products_path)
aisles = pd.read_csv(aisles_path)
departments = pd.read_csv(departments_path)

print("Orders:", orders.shape)
print("Products:", products.shape)
print("Aisles:", aisles.shape)
print("Departments:", departments.shape)

Orders: (3421083, 6)
Products: (49688, 4)
Aisles: (134, 2)
Departments: (21, 2)


In [67]:
order_products_preview = pd.read_csv(order_products_path, nrows=10)

order_products_preview

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1
5,1,13176,6,0
6,1,47209,7,0
7,1,22035,8,1
8,2,33120,1,1
9,2,28985,2,1


In [68]:
sample_size = 50000

sampled_orders = orders.sample(n=sample_size, random_state=42)

print("Sampled orders:", sampled_orders.shape)

sampled_order_ids = set(sampled_orders["order_id"])

Sampled orders: (50000, 6)


In [69]:
order_products = pd.read_csv(order_products_path)

order_products_subset = order_products[
    order_products["order_id"].isin(sampled_order_ids)
]

print("Filtered order_products:", order_products_subset.shape)

Filtered order_products: (495774, 4)


In [70]:
order_products_subset = order_products_subset.merge(
    products,
    on="product_id"
)

order_products_subset.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,47,16797,1,1,Strawberries,24,4
1,47,39275,2,1,Organic Blueberries,123,4
2,47,43352,3,1,Raspberries,32,4
3,47,46041,4,0,Beef Franks,106,12
4,47,29223,5,0,Golden Pineapple,24,4


In [71]:
order_products_subset = order_products_subset.merge(
    aisles,
    on="aisle_id"
)

order_products_subset.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle
0,47,16797,1,1,Strawberries,24,4,fresh fruits
1,47,39275,2,1,Organic Blueberries,123,4,packaged vegetables fruits
2,47,43352,3,1,Raspberries,32,4,packaged produce
3,47,46041,4,0,Beef Franks,106,12,hot dogs bacon sausage
4,47,29223,5,0,Golden Pineapple,24,4,fresh fruits


In [72]:
transactions = order_products_subset.groupby("order_id")["aisle"].apply(list)

print("Total transactions:", len(transactions))

transactions.iloc[0]

Total transactions: 48938


['fresh fruits',
 'packaged vegetables fruits',
 'packaged produce',
 'hot dogs bacon sausage',
 'fresh fruits']

In [73]:
transactions = order_products_subset.groupby("order_id")["aisle"].apply(
    lambda x: sorted(set(x))
)

transactions.iloc[0]

['fresh fruits',
 'hot dogs bacon sausage',
 'packaged produce',
 'packaged vegetables fruits']

In [74]:
transactions.shape
transactions.iloc[0:5]

order_id
47     [fresh fruits, hot dogs bacon sausage, package...
322    [fresh herbs, fresh vegetables, nuts seeds dri...
360    [baking ingredients, condiments, fresh dips ta...
393    [breakfast bars pastries, candy chocolate, fre...
441    [asian foods, body lotions soap, cleaning prod...
Name: aisle, dtype: object

In [75]:
from mlxtend.preprocessing import TransactionEncoder

In [76]:
te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

df_transactions = pd.DataFrame(te_array, columns=te.columns_)

df_transactions.head()

,air fresheners candles,asian foods,baby accessories,baby bath body care,baby food formula,bakery desserts,baking ingredients,baking supplies decor,beauty,beers coolers,...,spreads,tea,tofu meat alternatives,tortillas flat bread,trail mix snack mix,trash bags liners,vitamins supplements,water seltzer sparkling water,white wines,yogurt
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [77]:
from mlxtend.frequent_patterns import apriori

In [78]:
frequent_itemsets = apriori(
    df_transactions,
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets.sort_values("support", ascending=False).head(10)

,support,itemsets
32,0.556030,frozenset({fresh fruits})
35,0.444951,frozenset({fresh vegetables})
68,0.367935,frozenset({packaged vegetables fruits})
544,0.317463,"frozenset({fresh fruits, fresh vegetables})"
574,0.270015,"frozenset({fresh fruits, packaged vegetables f..."
94,0.261842,frozenset({yogurt})
58,0.241653,frozenset({milk})
642,0.234746,"frozenset({fresh vegetables, packaged vegetabl..."
65,0.231252,frozenset({packaged cheese})
93,0.192264,frozenset({water seltzer sparkling water})


In [79]:
from mlxtend.frequent_patterns import association_rules

In [80]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)

rules.sort_values("confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
3074,"frozenset({fresh fruits, canned jarred vegetab...",frozenset({fresh vegetables}),0.014447,0.444951,0.013629,0.943423,2.120286,1.0,0.007201,9.810495,0.536111,0.030575,0.898068,0.487027
3123,"frozenset({fresh herbs, canned jarred vegetabl...",frozenset({fresh vegetables}),0.011402,0.444951,0.010687,0.937276,2.106471,1.0,0.005614,8.849071,0.531331,0.023980,0.886994,0.480647
6111,"frozenset({fresh dips tapenades, yogurt, packa...",frozenset({fresh fruits}),0.016817,0.556030,0.015673,0.931956,1.676090,1.0,0.006322,6.524772,0.410273,0.028129,0.846738,0.480072
5875,"frozenset({yogurt, packaged vegetables fruits,...",frozenset({fresh fruits}),0.016143,0.556030,0.014999,0.929114,1.670978,1.0,0.006023,6.263147,0.408137,0.026919,0.840336,0.478044
4099,"frozenset({fresh fruits, soup broth bouillon, ...",frozenset({fresh vegetables}),0.012546,0.444951,0.011647,0.928339,2.086385,1.0,0.006065,7.745460,0.527318,0.026124,0.870892,0.477258
6093,"frozenset({fresh dips tapenades, yogurt, packa...",frozenset({fresh fruits}),0.011545,0.556030,0.010687,0.925664,1.664773,1.0,0.004268,5.972453,0.403981,0.019191,0.832565,0.472442
6408,"frozenset({yogurt, packaged vegetables fruits,...",frozenset({fresh fruits}),0.012056,0.556030,0.011157,0.925424,1.664341,1.0,0.004453,5.953233,0.404033,0.020033,0.832024,0.472745
6132,"frozenset({fresh fruits, packaged vegetables f...",frozenset({fresh vegetables}),0.016511,0.444951,0.015244,0.923267,2.074988,1.0,0.007897,7.233545,0.526767,0.034162,0.861755,0.478763
6777,"frozenset({fresh vegetables, packaged cheese, ...",frozenset({fresh fruits}),0.017553,0.556030,0.016204,0.923166,1.660282,1.0,0.006444,5.778336,0.404798,0.029072,0.826940,0.476155
5100,"frozenset({frozen produce, fresh herbs, packag...",frozenset({fresh vegetables}),0.011811,0.444951,0.010891,0.922145,2.072466,1.0,0.005636,7.129299,0.523668,0.024427,0.859734,0.473311


In [81]:
frequent_itemsets.head()

,support,itemsets
0,0.042421,frozenset({asian foods})
1,0.045690,frozenset({baby food formula})
2,0.010340,frozenset({bakery desserts})
3,0.078814,frozenset({baking ingredients})
4,0.011545,frozenset({body lotions soap})


In [82]:
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({asian foods}),frozenset({fresh fruits}),0.042421,0.556030,0.027565,0.649807,1.168655,1.0,0.003978,1.267788,0.150709,0.048285,0.211224,0.349691
1,frozenset({asian foods}),frozenset({fresh vegetables}),0.042421,0.444951,0.027565,0.649807,1.460403,1.0,0.008690,1.584982,0.329223,0.059950,0.369078,0.355880
2,frozenset({asian foods}),frozenset({packaged cheese}),0.042421,0.231252,0.012935,0.304913,1.318534,1.0,0.003125,1.105975,0.252284,0.049608,0.095820,0.180423
3,frozenset({asian foods}),frozenset({packaged vegetables fruits}),0.042421,0.367935,0.021313,0.502408,1.365482,1.0,0.005705,1.270249,0.279515,0.054782,0.212753,0.280167
4,frozenset({asian foods}),frozenset({yogurt}),0.042421,0.261842,0.013916,0.328035,1.252799,1.0,0.002808,1.098507,0.210726,0.047927,0.089673,0.190590


In [83]:
support_results = []

for support in [0.02, 0.01, 0.005]:
    frequent_itemsets_tmp = apriori(
        df_transactions,
        min_support=support,
        use_colnames=True,
        max_len=2 # otherwise it takes so long to compile, because 0.005 is already a low min support value.
    )
    
    rules_tmp = association_rules(
        frequent_itemsets_tmp,
        metric="confidence",
        min_threshold=0.3
    )
    
    support_results.append({
        "min_support": support,
        "num_frequent_itemsets": len(frequent_itemsets_tmp),
        "num_rules": len(rules_tmp)
    })

support_results

[{'min_support': 0.02, 'num_frequent_itemsets': 420, 'num_rules': 266},
 {'min_support': 0.01, 'num_frequent_itemsets': 930, 'num_rules': 378},
 {'min_support': 0.005, 'num_frequent_itemsets': 1747, 'num_rules': 459}]

In [84]:
confidence_results = []

frequent_itemsets_base = apriori(
    df_transactions,
    min_support=0.01,
    use_colnames=True,
    max_len=2 # I had to limit max_len for support_results, so for consistency I'm doing the same here.
)

for confidence in [0.2, 0.3, 0.5, 0.7]:
    rules_tmp = association_rules(
        frequent_itemsets_base,
        metric="confidence",
        min_threshold=confidence
    )
    
    confidence_results.append({
        "min_confidence": confidence,
        "num_rules": len(rules_tmp)
    })

confidence_results

[{'min_confidence': 0.2, 'num_rules': 639},
 {'min_confidence': 0.3, 'num_rules': 378},
 {'min_confidence': 0.5, 'num_rules': 159},
 {'min_confidence': 0.7, 'num_rules': 17}]

In [85]:
frequent_itemsets_final = apriori(
    df_transactions,
    min_support=0.01,
    use_colnames=True,
)

rules_final = association_rules(
    frequent_itemsets_final,
    metric="confidence",
    min_threshold=0.3
)

rules_single = rules_final[
    (rules_final["antecedents"].apply(len) == 1) &
    (rules_final["consequents"].apply(len) == 1)
].copy()

rules_single_filtered = rules_single[
    (rules_single["support"] >= 0.01) &
    (rules_single["confidence"] >= 0.3) &
    (rules_single["lift"] > 1.0)
].copy()

rules_single_filtered.sort_values(["lift", "confidence"], ascending=[False, False]).head(15)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
123,frozenset({pasta sauce}),frozenset({dry pasta}),0.062303,0.070518,0.019555,0.313873,4.450981,1.0,0.015162,1.354680,0.826846,0.172650,0.261818,0.295592
81,frozenset({preserved dips spreads}),frozenset({chips pretzels}),0.031877,0.170420,0.014488,0.454487,2.666870,1.0,0.009055,1.520735,0.645609,0.077141,0.342423,0.269750
71,frozenset({fresh dips tapenades}),frozenset({chips pretzels}),0.099718,0.170420,0.036066,0.361680,2.122292,1.0,0.019072,1.299631,0.587384,0.154081,0.230551,0.286656
285,frozenset({granola}),frozenset({yogurt}),0.028465,0.261842,0.015652,0.549892,2.100096,1.0,0.008199,1.639960,0.539179,0.056990,0.390229,0.304835
69,frozenset({cookies cakes}),frozenset({chips pretzels}),0.058114,0.170420,0.020495,0.352672,2.069434,1.0,0.010591,1.281546,0.548661,0.098517,0.219693,0.236468
80,frozenset({popcorn jerky}),frozenset({chips pretzels}),0.045486,0.170420,0.015918,0.349955,2.053489,1.0,0.008166,1.276189,0.537472,0.079595,0.216417,0.221680
18,frozenset({lunch meat}),frozenset({bread}),0.106277,0.162777,0.035167,0.330898,2.032825,1.0,0.017867,1.251263,0.568491,0.150358,0.200807,0.273471
372,frozenset({tofu meat alternatives}),frozenset({soy lactosefree}),0.032572,0.170542,0.010994,0.337516,1.979073,1.0,0.005439,1.252041,0.511369,0.057222,0.201304,0.200989
330,frozenset({pasta sauce}),frozenset({packaged cheese}),0.062303,0.231252,0.028505,0.457527,1.978480,1.0,0.014098,1.417118,0.527422,0.107548,0.294342,0.290396
76,frozenset({fruit vegetable snacks}),frozenset({chips pretzels}),0.046365,0.170420,0.015489,0.334068,1.960265,1.0,0.007588,1.245743,0.513682,0.076947,0.197266,0.212478


In [86]:
rules_single_filtered.sort_values("confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
200,frozenset({fresh herbs}),frozenset({fresh vegetables}),0.092648,0.444951,0.077813,0.839876,1.887572,1.0,0.036589,3.466382,0.518232,0.169237,0.711515,0.507378
52,frozenset({canned jarred vegetables}),frozenset({fresh vegetables}),0.075381,0.444951,0.057910,0.768230,1.726550,1.0,0.024369,2.394827,0.455118,0.125232,0.582433,0.449190
143,frozenset({fresh herbs}),frozenset({fresh fruits}),0.092648,0.556030,0.070150,0.757168,1.361739,1.0,0.018635,1.828301,0.292770,0.121256,0.453044,0.441665
240,frozenset({poultry counter}),frozenset({fresh vegetables}),0.037864,0.444951,0.028669,0.757151,1.701650,1.0,0.011821,2.285569,0.428562,0.063127,0.562472,0.410791
224,frozenset({meat counter}),frozenset({fresh vegetables}),0.020434,0.444951,0.015469,0.757000,1.701312,1.0,0.006376,2.284153,0.420817,0.034381,0.562201,0.395882
5,frozenset({baby food formula}),frozenset({fresh fruits}),0.045690,0.556030,0.033594,0.735242,1.322305,1.0,0.008188,1.676886,0.255415,0.059130,0.403657,0.397829
176,frozenset({packaged vegetables fruits}),frozenset({fresh fruits}),0.367935,0.556030,0.270015,0.733866,1.319832,1.0,0.065432,1.668223,0.383391,0.412899,0.400560,0.609739
212,frozenset({grains rice dried goods}),frozenset({fresh vegetables}),0.041522,0.444951,0.030385,0.731791,1.644657,1.0,0.011910,2.069468,0.408951,0.066622,0.516784,0.400040
252,frozenset({tofu meat alternatives}),frozenset({fresh vegetables}),0.032572,0.444951,0.023806,0.730866,1.642577,1.0,0.009313,2.062351,0.404372,0.052468,0.515116,0.392184
151,frozenset({frozen produce}),frozenset({fresh fruits}),0.124361,0.556030,0.090809,0.730200,1.313239,1.0,0.021660,1.645555,0.272400,0.154022,0.392302,0.446758


In [87]:
rules_single_filtered.sort_values("support", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
144,frozenset({fresh fruits}),frozenset({fresh vegetables}),0.556030,0.444951,0.317463,0.570946,1.283166,1.0,0.070057,1.293657,0.497055,0.464454,0.226998,0.642212
145,frozenset({fresh vegetables}),frozenset({fresh fruits}),0.444951,0.556030,0.317463,0.713479,1.283166,1.0,0.070057,1.549518,0.397582,0.464454,0.354638,0.642212
176,frozenset({packaged vegetables fruits}),frozenset({fresh fruits}),0.367935,0.556030,0.270015,0.733866,1.319832,1.0,0.065432,1.668223,0.383391,0.412899,0.400560,0.609739
175,frozenset({fresh fruits}),frozenset({packaged vegetables fruits}),0.556030,0.367935,0.270015,0.485612,1.319832,1.0,0.065432,1.228772,0.545821,0.412899,0.186179,0.609739
235,frozenset({fresh vegetables}),frozenset({packaged vegetables fruits}),0.444951,0.367935,0.234746,0.527577,1.433888,1.0,0.071033,1.337923,0.545169,0.406037,0.252573,0.582794
234,frozenset({packaged vegetables fruits}),frozenset({fresh vegetables}),0.367935,0.444951,0.234746,0.638010,1.433888,1.0,0.071033,1.533326,0.478741,0.406037,0.347823,0.582794
198,frozenset({fresh fruits}),frozenset({yogurt}),0.556030,0.261842,0.186481,0.335379,1.280848,1.0,0.040889,1.110646,0.493878,0.295349,0.099623,0.523784
199,frozenset({yogurt}),frozenset({fresh fruits}),0.261842,0.556030,0.186481,0.712190,1.280848,1.0,0.040889,1.542579,0.297046,0.295349,0.351735,0.523784
167,frozenset({milk}),frozenset({fresh fruits}),0.241653,0.556030,0.163043,0.674700,1.213423,1.0,0.028677,1.364801,0.231932,0.256906,0.267292,0.483963
172,frozenset({packaged cheese}),frozenset({fresh fruits}),0.231252,0.556030,0.155666,0.673147,1.210630,1.0,0.027083,1.358315,0.226321,0.246457,0.263794,0.476553
